# FA19 → COX-2 Re-Docking (AutoDock Vina)
### Cross-validating the MOE lead against an open-source force field

**Project:** Rational Design of a Novel Ferulic Acid Derivative Targeting COX-2 in Breast Cancer
**Phase:** TEST / PREDICT (DMTA)

This notebook independently re-docks the lead candidate **FA19**
(*4-(2-carboxyvinyl)-2-methoxyphenyl 4-methoxybenzoate*, C₁₈H₁₆O₆) into the
COX-2 active site using **AutoDock Vina**, to check whether the commercial
**MOE** result (**−8.3 kcal/mol**) reproduces on an open-source stack.

**It is fully self-contained** — no file uploads required. It will:
1. Fetch the COX-2 crystal structure **3LN1** (COX-2 + celecoxib) from the RCSB PDB.
2. Derive the docking box automatically from the co-crystallized celecoxib.
3. Prepare receptor and FA19 ligand to PDBQT.
4. Run Vina and report binding affinity vs. the MOE benchmark.
5. **Validate the protocol** by re-docking celecoxib and measuring pose RMSD to the crystal.

> Runtime → CPU is fine (Vina does not need a GPU). Runtime: Run all.


## 1 · Install dependencies

In [ ]:
# AutoDock Vina (python bindings), Meeko (ligand prep), RDKit, Open Babel
!pip -q install vina meeko rdkit numpy requests 2>/dev/null
!apt-get -qq install -y openbabel >/dev/null 2>&1
print("Dependencies installed.")

## 2 · Configuration
Everything you might want to change lives here.

In [ ]:
# ---- Target ----
PDB_ID      = "3LN1"          # COX-2 + celecoxib. Human alternative: "5KIR" (COX-2 + rofecoxib)
CHAIN       = "A"            # 3LN1 is a homodimer; dock into one monomer
NATIVE_LIG  = "CEL"          # 3-letter code of the co-crystal ligand (celecoxib). For 5KIR use "RCX"

# ---- Lead candidate (FA19) ----
FA19_SMILES = "COc1ccc(C(=O)Oc2ccc(/C=C/C(=O)O)cc2OC)cc1"
FA19_NAME   = "FA19"

# ---- Docking box / search ----
BOX_SIZE      = 25.0         # cubic box edge in Angstrom (FA19 is fairly extended)
EXHAUSTIVENESS = 16          # higher = more thorough / slower
N_POSES        = 10
SEED           = 42

# ---- Benchmark ----
MOE_AFFINITY  = -8.3         # kcal/mol, from MOE for FA19
print("Config loaded for", FA19_NAME, "->", PDB_ID)

## 3 · Fetch the receptor and locate the active site
The grid box is centred on the **centroid of the co-crystallized celecoxib** — an objective, reproducible definition of the COX-2 binding site.

In [ ]:
import requests, numpy as np

pdb_path = f"{PDB_ID}.pdb"
url = f"https://files.rcsb.org/download/{PDB_ID}.pdb"
open(pdb_path, "w").write(requests.get(url, timeout=60).text)
print("Downloaded", pdb_path)

# Collect coordinates of the native ligand in the chosen chain -> box center
lig_xyz = []
for line in open(pdb_path):
    if line.startswith("HETATM") and line[17:20].strip() == NATIVE_LIG and line[21] == CHAIN:
        lig_xyz.append([float(line[30:38]), float(line[38:46]), float(line[46:54])])
lig_xyz = np.array(lig_xyz)
assert len(lig_xyz) > 0, f"Native ligand {NATIVE_LIG} not found in chain {CHAIN}"
center = lig_xyz.mean(axis=0).round(3).tolist()
print(f"{NATIVE_LIG} atoms found: {len(lig_xyz)}")
print("Box center (x, y, z):", center)
print("Box size:", BOX_SIZE)

## 4 · Prepare the receptor (→ PDBQT)
Keep one protein chain, strip waters / cofactors / the native ligand, add hydrogens at pH 7.4.

In [ ]:
# Write a clean, protein-only PDB (chain CHAIN, ATOM records only)
clean = "receptor_clean.pdb"
with open(clean, "w") as out:
    for line in open(pdb_path):
        if line.startswith("ATOM") and line[21] == CHAIN:
            out.write(line)
    out.write("TER\nEND\n")

# Rigid receptor PDBQT via Open Babel (-xr), protonated at pH 7.4 (-p)
!obabel {clean} -O receptor.pdbqt -xr -p 7.4 2>/dev/null
import os
print("receptor.pdbqt written:", os.path.getsize("receptor.pdbqt"), "bytes")

## 5 · Prepare the FA19 ligand (→ PDBQT)
3D structure from SMILES (RDKit, MMFF-minimised), then converted to PDBQT with Meeko, with an Open Babel fallback.

In [ ]:
from rdkit import Chem
from rdkit.Chem import AllChem

m = Chem.AddHs(Chem.MolFromSmiles(FA19_SMILES))
AllChem.EmbedMolecule(m, randomSeed=SEED)
AllChem.MMFFOptimizeMolecule(m)
Chem.MolToMolFile(m, "fa19.mol")

ligand_pdbqt = "fa19.pdbqt"
try:
    from meeko import MoleculePreparation, PDBQTWriterLegacy
    prep = MoleculePreparation()
    setups = prep.prepare(m)
    pdbqt_str, ok, err = PDBQTWriterLegacy.write_string(setups[0])
    assert ok, err
    open(ligand_pdbqt, "w").write(pdbqt_str)
    print("Ligand prepared with Meeko.")
except Exception as e:
    print("Meeko path failed (", e, ") -> Open Babel fallback")
    !obabel fa19.mol -O {ligand_pdbqt} -p 7.4 2>/dev/null
import os; print(ligand_pdbqt, ":", os.path.getsize(ligand_pdbqt), "bytes")

## 6 · Dock FA19

In [ ]:
from vina import Vina

v = Vina(sf_name="vina", cpu=0, seed=SEED, verbosity=1)
v.set_receptor("receptor.pdbqt")
v.set_ligand_from_file("fa19.pdbqt")
v.compute_vina_maps(center=center, box_size=[BOX_SIZE]*3)
v.dock(exhaustiveness=EXHAUSTIVENESS, n_poses=N_POSES)
v.write_poses("fa19_poses.pdbqt", n_poses=N_POSES, overwrite=True)

energies = v.energies(n_poses=N_POSES)
fa19_best = float(energies[0][0])
print("FA19 docked. Poses (kcal/mol):", [round(float(e[0]), 2) for e in energies])

## 7 · Result vs. the MOE benchmark

In [ ]:
delta = fa19_best - MOE_AFFINITY
print(f"{'FA19 best Vina affinity':28}: {fa19_best:6.2f} kcal/mol")
print(f"{'MOE reference (FA19)':28}: {MOE_AFFINITY:6.2f} kcal/mol")
print(f"{'Difference':28}: {delta:+6.2f} kcal/mol")
print()
if abs(delta) <= 1.0:
    print("CROSS-VALIDATED: agreement within ~1 kcal/mol across scoring functions.")
else:
    print("DISCREPANCY > 1 kcal/mol: note box/protonation/SF differences in your write-up.")

## 8 · Protocol validation — celecoxib redock (positive control)
Re-docking the native ligand and recovering its crystallographic pose
(**RMSD < 2 Å**) is the standard evidence that the docking setup is trustworthy.
Without this, a good FA19 score means little.

In [ ]:
# Extract crystal celecoxib (chain A) and convert to SDF
with open("cel_crystal.pdb", "w") as out:
    for line in open(pdb_path):
        if line.startswith("HETATM") and line[17:20].strip()==NATIVE_LIG and line[21]==CHAIN:
            out.write(line)
    out.write("END\n")
!obabel cel_crystal.pdb -O cel_crystal.sdf 2>/dev/null
!obabel cel_crystal.pdb -O cel.pdbqt -p 7.4 2>/dev/null

vc = Vina(sf_name="vina", cpu=0, seed=SEED, verbosity=0)
vc.set_receptor("receptor.pdbqt")
vc.set_ligand_from_file("cel.pdbqt")
vc.compute_vina_maps(center=center, box_size=[BOX_SIZE]*3)
vc.dock(exhaustiveness=EXHAUSTIVENESS, n_poses=N_POSES)
vc.write_poses("cel_redock.pdbqt", n_poses=1, overwrite=True)
print("Celecoxib redock affinity:", round(float(vc.energies(1)[0][0]),2), "kcal/mol")

# Top docked pose -> SDF, then RMSD vs crystal with obrms (handles symmetry)
!obabel cel_redock.pdbqt -O cel_redock.sdf -f 1 -l 1 2>/dev/null
print("\\nPose RMSD to crystal:")
!obrms cel_crystal.sdf cel_redock.sdf

## 9 · Notes & how to adapt
- **Swap the target:** set `PDB_ID="5KIR"`, `NATIVE_LIG="RCX"` for human COX-2 + rofecoxib.
- **Use your own prepared files:** skip cells 3–5 and point `set_receptor` / `set_ligand_from_file` at your PDBQTs; set `center` manually.
- **Reproducibility:** `SEED` and `EXHAUSTIVENESS` are fixed so reruns are comparable. Report both in your methods.
- **Interpretation:** Vina and MOE use different scoring functions, so agreement within ~1 kcal/mol is the goal — not an identical number. The celecoxib positive control (Section 8) is what validates the protocol itself.
- **Outputs to keep for the portfolio:** `fa19_poses.pdbqt`, the affinity table, and the redock RMSD.
